# Лабораторная работа 2. Формирование отчётов в Apache Spark

**Задание:** Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

## Установка Spark



In [1]:
!wget http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install findspark

--2026-03-11 13:22:43--  http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz.1’

spark-3.5.1-bin-had 100%[===================>] 381.90M  9.96MB/s    in 33s     

2026-03-11 13:23:16 (11.6 MB/s) - ‘spark-3.5.1-bin-hadoop3.tgz.1’ saved [400446614/400446614]



In [2]:
import os
import sys
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[*]")
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.18.0")
    .getOrCreate()
)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

In [5]:
from google.colab import files
uploaded = files.upload()

Saving posts_sample.xml to posts_sample (1).xml
Saving programming-languages.csv to programming-languages (1).csv


## Чтение XML-файла в DataFrame

In [6]:
'''
# Загружаем XML файл как RDD строки
raw_rdd = spark.sparkContext.textFile("posts_sample.xml")
raw_count = raw_rdd.count()
posts_raw_data = (
    raw_rdd.zipWithIndex()
    .filter(lambda x: x[1] >= 2 and x[1] < raw_count - 1)
    .map(lambda x: x[0])
)'''
# Сразу читаем posts_sample.xml как таблицу Spark. Каждый тег <row ... /> - строка DataFrame
posts_data = spark.read.format("xml").option("rowTag", "row").option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss.SSS").load("posts_sample.xml")
posts_data.printSchema()
posts_data.show(5)

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: timestamp (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: timestamp (nullable = true)
 |-- _CreationDate: timestamp (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: timestamp (nullable = true)
 |-- _LastEditDate: timestamp (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)

+-----------------+------------+--------------------+-----------+------------

## Фильтрация постов по датам 2010-2020

In [11]:
from pyspark.sql import functions as F

posts_filtered = (posts_data.filter(F.col("_CreationDate").isNotNull()
    & (F.year("_CreationDate") >= 2010)
    & (F.year("_CreationDate") <= 2020))
)
posts_filtered.select("_Id", "_CreationDate").show(10)
print("Количество постов за 2010-2020 годы:", posts_filtered.count())

+-------+--------------------+
|    _Id|       _CreationDate|
+-------+--------------------+
|3753373|2010-09-20 16:18:...|
|3754384|2010-09-20 18:36:...|
|3754601|2010-09-20 19:04:...|
|3756831|2010-09-21 02:07:...|
|3758183|2010-09-21 07:33:...|
|3759958|2010-09-21 11:46:...|
|3760101|2010-09-21 12:07:...|
|3760630|2010-09-21 13:16:...|
|3761678|2010-09-21 15:09:...|
|3762339|2010-09-21 16:22:...|
+-------+--------------------+
only showing top 10 rows

Количество постов за 2010-2020 годы: 44419


## Сохранение XML в Parquet

In [12]:
posts_filtered.write.mode("overwrite").parquet("posts_2010_2020_parquet")

In [13]:
# читаем Parquet обратно
posts_filtered_parquet = spark.read.parquet("posts_2010_2020_parquet")
posts_filtered_parquet.show(10)

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|_Tags|_Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|             NULL|        NULL|<p>No. (And more ...|       NULL

## Чтение CSV-файла с языками программирования в DataFrame

In [16]:
languages_df = (spark.read.format("csv").option("header", True).option("inferSchema", True).load("programming-languages.csv"))
languages_df.printSchema()
languages_df.show(5)

root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)

+----------+--------------------+
|      name|       wikipedia_url|
+----------+--------------------+
|   A# .NET|https://en.wikipe...|
|A# (Axiom)|https://en.wikipe...|
|A-0 System|https://en.wikipe...|
|        A+|https://en.wikipe...|
|       A++|https://en.wikipe...|
+----------+--------------------+
only showing top 5 rows



## Подсчет популярности языков программирования по годам

In [19]:
# оставляем только записи с тегами и годами
tags_df = (posts_filtered.withColumn("_Year", F.year("_CreationDate"))
          .filter(F.col("_Tags").isNotNull())
          .select("_Id", "_Year", "_Tags", "_Title", "_Body")
)
tags_df.show(10)

+-------+-----+--------------------+--------------------+--------------------+
|    _Id|_Year|               _Tags|              _Title|               _Body|
+-------+-----+--------------------+--------------------+--------------------+
|3768363| 2010|<c++><character-e...|Character sets - ...|<p>The standard d...|
|3775996| 2010|<sharepoint><info...|Hiow to switch In...|<p>I am using the...|
|3776721| 2010|<iphone><app-stor...|InApp Purchase re...|<p>I just got rej...|
|3777993| 2010|<symfony1><schema...|Doctrine fixtures...|<p>Is there any w...|
|3778222| 2010|              <java>|Java text highliting|<p>I did a code i...|
|3785460| 2010|<visual-studio-20...|Split Stylecop ru...|<p>I like to spli...|
|3789116| 2010|<cakephp><file-up...|Best/Fastest way ...|<p>My users need ...|
|3794815| 2010|<git><cygwin><putty>|Git - Authenticat...|<p>I'm completely...|
|3797876| 2010|  <drupal><drupal-6>|Can I install Dru...|<p>Can I install ...|
|3798872| 2010|<php><wordpress><...|Wordpress - Tryi

In [33]:
from pyspark.sql.functions import regexp_replace

# удаляем символы < > из тегов
posts_clean = (tags_df.withColumn("_Tags", regexp_replace(F.col("_Tags"), "[<>]", " "))
              .withColumn("_TagArray", F.split(F.col("_Tags"), r"\s+")))
# делаем каждую строку отдельной записью
posts_clean = (posts_clean.withColumn("_Language", F.explode(F.col("_TagArray")))
              .select("_Id", "_Year", "_Language", "_Title", "_Body")
              .filter(F.col("_Language") != ""))
posts_clean.show(10)

+-------+-----+------------------+--------------------+--------------------+
|    _Id|_Year|         _Language|              _Title|               _Body|
+-------+-----+------------------+--------------------+--------------------+
|3768363| 2010|               c++|Character sets - ...|<p>The standard d...|
|3768363| 2010|character-encoding|Character sets - ...|<p>The standard d...|
|3775996| 2010|        sharepoint|Hiow to switch In...|<p>I am using the...|
|3775996| 2010|          infopath|Hiow to switch In...|<p>I am using the...|
|3776721| 2010|            iphone|InApp Purchase re...|<p>I just got rej...|
|3776721| 2010|         app-store|InApp Purchase re...|<p>I just got rej...|
|3776721| 2010|   in-app-purchase|InApp Purchase re...|<p>I just got rej...|
|3777993| 2010|          symfony1|Doctrine fixtures...|<p>Is there any w...|
|3777993| 2010|            schema|Doctrine fixtures...|<p>Is there any w...|
|3777993| 2010|          doctrine|Doctrine fixtures...|<p>Is there any w...|

In [45]:
# оставляем только теги-языки программирования
language_names = [row["name"].lower() for row in languages_df.collect()]
language_posts_df = (posts_clean.filter(F.col("_Language").isin(language_names)))
language_posts_df.show(10)

+-------+-----+-----------+--------------------+--------------------+
|    _Id|_Year|  _Language|              _Title|               _Body|
+-------+-----+-----------+--------------------+--------------------+
|3778222| 2010|       java|Java text highliting|<p>I did a code i...|
|3798872| 2010|        php|Wordpress - Tryin...|<p>Ah I'm despera...|
|3833655| 2010|       ruby|How do I rename a...|<p>Is there an RV...|
|3838866| 2010|          c|What is the sizeo...|<p>In the gcc com...|
|3859099| 2010|        php|Getting the visit...|<p>I'm doing a pr...|
|3872977| 2010|     python|Running a job on ...|<p>I have access ...|
|3885802| 2010| javascript|jQuery start typi...|<p>I am making a ...|
|3886770| 2010|applescript|Applescript scrap...|<p>there is this ...|
|3891676| 2010|        php|PHP server side t...|<p>I want to crea...|
|3904423| 2010|        php|Embed PHP in HTML...|<p>In an effort t...|
+-------+-----+-----------+--------------------+--------------------+
only showing top 10 

In [50]:
language_popularity = (language_posts_df.groupBy("_Year", "_Language").agg(F.count("*").alias("_MentionCount"))
                      .orderBy("_Year", F.desc("_MentionCount")))
language_popularity.show(10)

+-----+-----------+-------------+
|_Year|  _Language|_MentionCount|
+-----+-----------+-------------+
| 2010|       java|           52|
| 2010|        php|           46|
| 2010| javascript|           44|
| 2010|     python|           26|
| 2010|objective-c|           23|
| 2010|          c|           20|
| 2010|       ruby|           12|
| 2010|     delphi|            8|
| 2010|applescript|            3|
| 2010|          r|            3|
+-----+-----------+-------------+
only showing top 10 rows



In [51]:
from pyspark.sql.window import Window

# создаем окно для ранжирования внутри каждого года
window_spec = Window.partitionBy("_Year").orderBy(F.desc("_MentionCount"))

top10_df = (language_popularity.withColumn("_Rank", F.row_number().over(window_spec))
            .filter(F.col("_Rank") <= 10).orderBy("_Year", "_Rank"))
top10_df.show(200)

+-----+-----------+-------------+-----+
|_Year|  _Language|_MentionCount|_Rank|
+-----+-----------+-------------+-----+
| 2010|       java|           52|    1|
| 2010|        php|           46|    2|
| 2010| javascript|           44|    3|
| 2010|     python|           26|    4|
| 2010|objective-c|           23|    5|
| 2010|          c|           20|    6|
| 2010|       ruby|           12|    7|
| 2010|     delphi|            8|    8|
| 2010|applescript|            3|    9|
| 2010|          r|            3|   10|
| 2011|        php|          102|    1|
| 2011|       java|           93|    2|
| 2011| javascript|           83|    3|
| 2011|     python|           37|    4|
| 2011|objective-c|           34|    5|
| 2011|          c|           24|    6|
| 2011|       ruby|           20|    7|
| 2011|       perl|            9|    8|
| 2011|     delphi|            8|    9|
| 2011|       bash|            7|   10|
| 2012|        php|          154|    1|
| 2012| javascript|          132|    2|


## Сохранение итогового отчета о популярности языков программирования по годам в Parquet

In [58]:
top10_df.write.mode("overwrite").parquet("top_languages_report_parquet")

In [61]:
# читаем Parquet обратно
top10_df_parquet = spark.read.parquet("top_languages_report_parquet")
top10_df_parquet.show(20)

+-----+-----------+-------------+-----+
|_Year|  _Language|_MentionCount|_Rank|
+-----+-----------+-------------+-----+
| 2010|       java|           52|    1|
| 2010|        php|           46|    2|
| 2010| javascript|           44|    3|
| 2010|     python|           26|    4|
| 2010|objective-c|           23|    5|
| 2010|          c|           20|    6|
| 2010|       ruby|           12|    7|
| 2010|     delphi|            8|    8|
| 2010|applescript|            3|    9|
| 2010|          r|            3|   10|
| 2011|        php|          102|    1|
| 2011|       java|           93|    2|
| 2011| javascript|           83|    3|
| 2011|     python|           37|    4|
| 2011|objective-c|           34|    5|
| 2011|          c|           24|    6|
| 2011|       ruby|           20|    7|
| 2011|       perl|            9|    8|
| 2011|     delphi|            8|    9|
| 2011|       bash|            7|   10|
+-----+-----------+-------------+-----+
only showing top 20 rows

